# Exercice 1 — Étude de cas NovaRetail (Bloc 2)

Notebook Jupyter **structuré** pour :
- sélectionner les variables utiles,
- calculer les KPI (CTR, taux de conversion, CPL, coût par lead/client),
- réaliser les analyses univariées et bivariées,
- exporter les résultats.

## 1) Imports et chemins
Ce notebook est autonome (pas de dépendances externes obligatoires).

In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from statistics import mean, median

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR)

## 2) Fonctions utilitaires

In [ ]:
def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

## 3) Chargement des données
- `leads_novaretail.csv`
- `campaign_novaretail.json`
- `crm_novaretail.csv`

In [ ]:
leads = read_csv(DATA_DIR / "leads_novaretail.csv")
crm = read_csv(DATA_DIR / "crm_novaretail.csv")
campaigns = json.loads((DATA_DIR / "campaign_novaretail.json").read_text(encoding="utf-8"))

print(f"Leads: {len(leads)} | CRM: {len(crm)} | Campagnes: {len(campaigns)}")

## 4) Périmètre d'analyse
Contrainte : **octobre 2025 uniquement**.

In [ ]:
start = datetime(2025, 10, 1)
end = datetime(2025, 10, 31)

leads_scope = []
for row in leads:
    d = datetime.strptime(row["date"], "%Y-%m-%d")
    if start <= d <= end:
        leads_scope.append(row)

print("Leads dans le périmètre:", len(leads_scope))

## 5) Jointure Leads + CRM
Clé de jointure : `lead_id`.

In [ ]:
crm_by_lead = {row["lead_id"]: row for row in crm}
merged = []
for lead in leads_scope:
    crm_row = crm_by_lead.get(lead["lead_id"])
    if crm_row:
        merged.append({**lead, **crm_row})

print("Lignes fusionnées:", len(merged))
merged[:2]

## 6) KPI par canal
Formules demandées :
- `CTR = clicks / impressions`
- `Taux conversion = conversions / clicks`
- `CPL = cost / conversions`

In [ ]:
kpi_rows = []
for camp in campaigns:
    row = dict(camp)
    row["ctr"] = row["clicks"] / row["impressions"]
    row["conversion_rate"] = row["conversions"] / row["clicks"]
    row["cpl"] = row["cost"] / row["conversions"]

    leads_count = sum(1 for x in merged if x["channel"] == row["channel"])
    clients_count = sum(1 for x in merged if x["channel"] == row["channel"] and x["status"] == "Client")
    row["leads"] = leads_count
    row["clients"] = clients_count
    row["cost_per_lead"] = row["cost"] / leads_count if leads_count else None
    row["cost_per_client"] = row["cost"] / clients_count if clients_count else None
    kpi_rows.append(row)

kpi_rows

## 7) Analyse univariée
- Quantitative : moyenne, médiane, min, max, dispersion (range).
- Qualitative : fréquences et proportions.

In [ ]:
quant_vars = ["cost", "impressions", "clicks", "conversions", "ctr", "conversion_rate", "cpl"]
quant_summary = []
for var in quant_vars:
    values = [float(r[var]) for r in kpi_rows]
    quant_summary.append(
        {
            "variable": var,
            "mean": mean(values),
            "median": median(values),
            "min": min(values),
            "max": max(values),
            "range": max(values) - min(values),
        }
    )

channel_counts = Counter(row["channel"] for row in merged)
status_counts = Counter(row["status"] for row in merged)

quant_summary, channel_counts, status_counts

## 8) Analyse bivariée
Croisements métier :
- canal × statut,
- taille d'entreprise × statut,
- secteur × statut,
- région × taux client.

In [ ]:
status_by_channel = defaultdict(int)
status_by_size = defaultdict(int)
status_by_sector = defaultdict(int)

for row in merged:
    status_by_channel[(row["channel"], row["status"])] += 1
    status_by_size[(row["company_size"], row["status"])] += 1
    status_by_sector[(row["sector"], row["status"])] += 1

channels = sorted({r["channel"] for r in merged})
statuses = sorted({r["status"] for r in merged})
sizes = sorted({r["company_size"] for r in merged})
sectors = sorted({r["sector"] for r in merged})

rows_channel = []
for c in channels:
    row = {"channel": c}
    for s in statuses:
        row[s] = status_by_channel[(c, s)]
    rows_channel.append(row)

rows_size = []
for c in sizes:
    row = {"company_size": c}
    for s in statuses:
        row[s] = status_by_size[(c, s)]
    rows_size.append(row)

rows_sector = []
for c in sectors:
    row = {"sector": c}
    for s in statuses:
        row[s] = status_by_sector[(c, s)]
    rows_sector.append(row)

region_totals = Counter(r["region"] for r in merged)
region_clients = Counter(r["region"] for r in merged if r["status"] == "Client")
region_rows = []
for region, total in sorted(region_totals.items(), key=lambda x: x[0]):
    region_rows.append(
        {
            "region": region,
            "leads": total,
            "clients": region_clients[region],
            "client_rate": region_clients[region] / total,
        }
    )

rows_channel, rows_size, rows_sector, region_rows

## 9) Export des résultats
Les mêmes fichiers que le script sont produits dans `outputs/`.

In [ ]:
write_csv(
    OUTPUT_DIR / "kpi_par_canal.csv",
    kpi_rows,
    [
        "campaign_id", "channel", "cost", "impressions", "clicks", "conversions",
        "ctr", "conversion_rate", "cpl", "leads", "clients", "cost_per_lead", "cost_per_client"
    ],
)
write_csv(OUTPUT_DIR / "analyse_univariee_quant.csv", quant_summary, list(quant_summary[0].keys()))
write_csv(
    OUTPUT_DIR / "frequences_channel.csv",
    [{"channel": k, "count": v, "proportion": v / len(merged)} for k, v in channel_counts.items()],
    ["channel", "count", "proportion"],
)
write_csv(
    OUTPUT_DIR / "frequences_status.csv",
    [{"status": k, "count": v, "proportion": v / len(merged)} for k, v in status_counts.items()],
    ["status", "count", "proportion"],
)
write_csv(OUTPUT_DIR / "croisement_status_channel.csv", rows_channel, ["channel", *statuses])
write_csv(OUTPUT_DIR / "croisement_status_taille.csv", rows_size, ["company_size", *statuses])
write_csv(OUTPUT_DIR / "croisement_status_secteur.csv", rows_sector, ["sector", *statuses])
write_csv(OUTPUT_DIR / "taux_client_region.csv", region_rows, ["region", "leads", "clients", "client_rate"])

print("Exports terminés:", OUTPUT_DIR)

## 10) Interprétation rapide
- Emailing : meilleur CTR et CPL le plus faible.
- LinkedIn Ads : meilleur taux de conversion.
- Distribution CRM observée : MQL (4), SQL (3), Client (3).

> Pour la restitution finale, utiliser ce notebook + les documents `reports/`.